# Run Evaluation Metrics (vanilla vs VUF vs SAE)

This notebook computes metrics on pre-generated `with_vufi_*.jsonl` files.

The core logic is taken from `calibration/eval/eval.ipynb` but extracted into Python cells so that multiple scenarios can be evaluated sequentially.

In [ ]:
# Устанавливаем зависимости для eval-скриптов и semantic entropy.
# Это нужно, чтобы ноутбук гарантированно запускался в Colab с чистого окружения.
import sys, subprocess

pkgs = [
    'transformers==4.48.0',
    'accelerate',
    'datasets',
    'evaluate==0.4.3',
    'peft==0.13.2',
    'safetensors',
    'tokenizers',
    'einops',
    'jsonlines',
    'tenacity',
    'openai',
    'scikit-learn',
    'scipy',
    'pandas',
    'tqdm',
    'submitit',
    'requests',
]

print('Installing python dependencies...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', '--no-cache-dir'] + pkgs)

print('Installing vllm...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', '--no-cache-dir', 'vllm'])

print('Dependencies installed.')


In [ ]:
import os, sys, re, json, shutil, subprocess
from pathlib import Path
import numpy as np

PROJECT_DIR = Path(os.getcwd()).resolve()
# Попытка сделать ноутбук более устойчивым: если CWD не внутри sae-muc-repo,
# то проверьте `./sae-muc-repo/eval_bundle` и переключите PROJECT_DIR.
if not (PROJECT_DIR / 'eval_bundle').exists():
    candidate = PROJECT_DIR / 'sae-muc-repo'
    if (candidate / 'eval_bundle').exists():
        PROJECT_DIR = candidate

CODE_DIR = PROJECT_DIR / 'eval_bundle'
CHECKPOINT_DIR = PROJECT_DIR.parent / 'vuf_checkpoint'  # где лежат datasets/, detection/, sem_uncertainty/ и calibration/

# Если вы клонировали только sae-muc-repo, `vuf_checkpoint/` может быть не загружен.
# Тогда ищем zip с bundle в текущей папке/родительской и распаковываем.
if not CHECKPOINT_DIR.exists():
    import zipfile

    zip_candidates = []
    for pat in [
        'vuf_muc_colab_bundle_full.zip',
        'vuf_muc_colab_bundle.zip',
        'vuf_muc_colab_bundle_sentence_only.zip',
    ]:
        zip_candidates += list(PROJECT_DIR.rglob(pat))
        zip_candidates += list(PROJECT_DIR.parent.rglob(pat))

    if zip_candidates:
        zip_path = zip_candidates[0]
        print('Found checkpoint zip:', zip_path)
        CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(CHECKPOINT_DIR)
        print('Checkpoint extracted to:', CHECKPOINT_DIR)
    else:
        raise RuntimeError(
            'Missing `vuf_checkpoint/`.\n'
            'Put `vuf_checkpoint` next to sae-muc-repo or upload one of the zips '
            '(vuf_muc_colab_bundle_full.zip / vuf_muc_colab_bundle.zip / vuf_muc_colab_bundle_sentence_only.zip) '
            'and re-run.'
        )

DATASET = 'nq_open'
MODEL_NAME = 'Mistral-7B-Instruct-v0.3'
SPLIT = 'test'
PROMPT_TYPE = 'uncertainty'

# Параметры, которые должны совпадать с именем вашего jsonl
ITI_METHOD = 2
STR_PROCESS_LAYERS = 'range(15,32)'
MAX_ALPHA = 1.0

# vLLM (LLM-as-judge) конфигурация — как в `colab_pipeline.ipynb`
START_VLLM = True  # поставьте False, если vLLM уже запущен
VLLM_PORT = 8000
VLLM_LOG = '/tmp/vllm.log'
JUDGE_HF_ID = 'mistralai/Mistral-7B-Instruct-v0.3'
JUDGE_ALIAS = 'meta-llama/Llama-3.1-70B-Instruct'
VLLM_URL = os.environ.get('VLLM_URL', f'http://localhost:{VLLM_PORT}/v1')
os.environ['VLLM_URL'] = VLLM_URL
ENTAILMENT_MODEL_URL = VLLM_URL
vllm_proc = None
if START_VLLM:
    import time, requests, gc
    import torch
    gc.collect()
    torch.cuda.empty_cache()

    vllm_proc = subprocess.Popen(
        [
            sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
            '--model', JUDGE_HF_ID,
            '--served-model-name', JUDGE_ALIAS,
            '--port', str(VLLM_PORT),
            '--gpu-memory-utilization', '0.85',
            '--max-model-len', '4096',
            '--dtype', 'float16',
        ],
        stdout=open(VLLM_LOG, 'w'),
        stderr=subprocess.STDOUT,
        cwd=str(PROJECT_DIR),
    )
    print(f"vLLM PID={vllm_proc.pid}. Логи: {VLLM_LOG}")

    for i in range(120):
        try:
            if requests.get(f'http://localhost:{VLLM_PORT}/health', timeout=3).status_code == 200:
                print(f'vLLM готов ({i*5}s)')
                break
        except Exception:
            pass
        print(f"  ожидание {(i+1)*5}s...", end='\r')
        time.sleep(5)
    else:
        raise RuntimeError('vLLM не стартовал. Логи смотрите в ' + VLLM_LOG)

# Сценарии: укажите пути к вашим jsonl (формат как в pipeline: question, most_likely_answer, responses, alpha)
SCENARIOS = {
    'vanilla': None,
    'vuf': None,
    'sae': None,
}

# Подсказка: если jsonl лежит рядом со скриптом/в проекте, можно указать абсолютный путь.
# Например:
# SCENARIOS['vuf'] = str(PROJECT_DIR / 'vuf_checkpoint' / 'calibration' / 'outputs' / 'nq_open' / 'Mistral-7B-Instruct-v0.3' / 'uncertainty' / 'test' / 'with_vufi_2_range(15,32)_1.0.jsonl')

print('PROJECT_DIR', PROJECT_DIR)
print('CODE_DIR', CODE_DIR)
print('CHECKPOINT_DIR', CHECKPOINT_DIR)
print('ENTAILMENT_MODEL_URL', ENTAILMENT_MODEL_URL)
print('SCENARIOS', {k:v for k,v in SCENARIOS.items()})

In [ ]:
from pathlib import Path

# Удобный вариант для Colab: загрузите 1-3 jsonl и ноутбук автоматически подставит пути.
try:
    from google.colab import files

    print("Upload jsonl files for scenarios (optional).")
    uploaded = files.upload(accept_multiple_files=True)
    uploaded_names = list(uploaded.keys())

    # Heuristics by filename substring
    for name in uploaded_names:
        low = name.lower()
        p = Path('/content') / name
        if 'vanilla' in low:
            SCENARIOS['vanilla'] = str(p)
        elif 'vuf' in low:
            SCENARIOS['vuf'] = str(p)
        elif 'sae' in low:
            SCENARIOS['sae'] = str(p)

    print('SCENARIOS after upload:', SCENARIOS)
except Exception as e:
    print('Upload cell skipped (likely not running in Colab):', repr(e))


In [ ]:
def symlink_or_copy(src: Path, dst: Path):
    if dst.exists() or dst.is_symlink():
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(src, dst)
    except Exception:
        if src.is_dir():
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)

# Подготовим именно данные, которые ожидают eval-скрипты.
# В репозитории уже есть `datasets/` и `detection/` (код), поэтому копируем конкретные файлы с checkpoint-а.
# 1) Датасет CSV нужен eval_acc.py как ground-truth.
src_ds = CHECKPOINT_DIR / 'datasets' / DATASET / MODEL_NAME / f'{SPLIT}.csv'
dst_ds = CODE_DIR / 'datasets' / DATASET / MODEL_NAME / f'{SPLIT}.csv'
symlink_or_copy(src_ds, dst_ds)

# 2) detection/LR_outputs нужен для метрики фильтрации (detection_res).
src_det = CHECKPOINT_DIR / 'detection' / 'LR_outputs' / DATASET / MODEL_NAME / f'{SPLIT}_verbal_uncertainty_sentence_semantic_entropy.json'
dst_det = CODE_DIR / 'detection' / 'LR_outputs' / DATASET / MODEL_NAME / f'{SPLIT}_verbal_uncertainty_sentence_semantic_entropy.json'
symlink_or_copy(src_det, dst_det)

# В `CODE_DIR/sem_uncertainty` уже лежит код. Нам нужны именно output-файлы из checkpoint.
sem_src = CHECKPOINT_DIR / 'sem_uncertainty'
sem_dst = CODE_DIR / 'sem_uncertainty'
for fname in ['test_most_likely_acc.json', 'test_refusal_rate.json', 'test_semantic_entropy.pkl']:
    src = sem_src / fname
    dst = sem_dst / fname
    if src.exists():
        symlink_or_copy(src, dst)

print('Symlinks/copies done.')

In [ ]:
base_out_dir = CODE_DIR / 'calibration' / 'outputs' / DATASET / MODEL_NAME / PROMPT_TYPE / SPLIT
base_out_dir.mkdir(parents=True, exist_ok=True)

input_jsonl_name = f'with_vufi_{ITI_METHOD}_{STR_PROCESS_LAYERS}_{MAX_ALPHA}.jsonl'
input_jsonl_path = base_out_dir / input_jsonl_name

acc_json = input_jsonl_path.with_name(input_jsonl_path.name.replace('with_vufi', 'acc').replace('.jsonl','') + '.json')
vu_json = input_jsonl_path.with_name(input_jsonl_path.name.replace('with_vufi', 'vu').replace('.jsonl','') + '.json')
vu_most_likely_json = input_jsonl_path.with_name(input_jsonl_path.name.replace('with_vufi', 'vu_most_likely').replace('.jsonl','') + '.json')
refusal_json = input_jsonl_path.with_name(input_jsonl_path.name.replace('with_vufi', 'refusal').replace('.jsonl','') + '.json')
se_pkl = input_jsonl_path.with_name(input_jsonl_path.name.replace('with_vufi', 'uncertainty_measures').replace('.jsonl','') + '.pkl')

print('input_jsonl_path', input_jsonl_path)
print('acc_json', acc_json)
print('vu_json', vu_json)
print('vu_most_likely_json', vu_most_likely_json)
print('refusal_json', refusal_json)
print('se_pkl', se_pkl)

In [ ]:
def best_split_threshold(values: np.ndarray) -> float:
    # Реализация best_split из paper repo без внешних импортов.
    values = np.asarray(values, dtype=float)
    splits = np.linspace(values.min(), values.max(), 100)
    split_mses = []
    for split in splits:
        low_mask = values < split
        high_mask = ~low_mask
        if not low_mask.any() or not high_mask.any():
            split_mses.append(np.inf)
            continue
        low_mean = float(values[low_mask].mean())
        high_mean = float(values[high_mask].mean())
        mse = float(np.sum((values[low_mask] - low_mean) ** 2) + np.sum((values[high_mask] - high_mean) ** 2))
        split_mses.append(mse)
    best_idx = int(np.argmin(np.array(split_mses, dtype=float)))
    return float(splits[best_idx])

def load_dataset_base():
    # В нашем vuf_checkpoint datasets/ может содержать только: id, question, answer, verbal_uncertainty, sentence_semantic_entropy
    import csv
    df = []
    with open(CODE_DIR / 'datasets' / DATASET / MODEL_NAME / f'{SPLIT}.csv', 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            df.append(row)
    ids = [r['id'] for r in df]
    qs = [r['question'] for r in df]
    lu = np.array([float(r['verbal_uncertainty']) for r in df], dtype=float)
    se = np.array([float(r['sentence_semantic_entropy']) for r in df], dtype=float)
    return ids, qs, lu, se

def load_before_accuracy_and_refusal():
    acc_path = CODE_DIR / 'sem_uncertainty' / 'test_most_likely_acc.json'
    ref_path = CODE_DIR / 'sem_uncertainty' / 'test_refusal_rate.json'
    missing = []
    if not acc_path.exists():
        missing.append(str(acc_path))
    if not ref_path.exists():
        missing.append(str(ref_path))
    if missing:
        raise RuntimeError(
            'Missing precomputed sem_uncertainty outputs for before-metrics. ' 
            'You need either full `vuf_checkpoint/sem_uncertainty/` directory, or copy these files into `eval_bundle/sem_uncertainty/`:\n' +
            ('\n'.join(missing))
        )
    with open(acc_path, 'r') as f:
        acc_map = json.load(f)
    with open(ref_path, 'r') as f:
        refusal_obj = json.load(f)
    refusal_list = refusal_obj['refusal']
    return acc_map, refusal_list

def compute_before_metrics():
    ids, qs, lu, se = load_dataset_base()
    acc_map, refusal_list = load_before_accuracy_and_refusal()
    acc = np.array([float(acc_map[i]) for i in ids], dtype=float)
    refusal = np.array([int(refusal_list[k]) for k in range(len(ids))], dtype=int)

    vu_th = best_split_threshold(lu)
    se_th = best_split_threshold(se)

    hallu_before = float(np.mean(acc == 0) * 100)
    correct_before = float(np.mean(acc == 1) * 100)
    correct_ex_refusal = float(np.mean((acc == 1) & (refusal == 0)) * 100)
    refusal_before = float(np.mean(refusal == 1) * 100)

    # disagreement по порогам как в eval.ipynb
    disagree = []
    for i in range(len(lu)):
        disagree.append(0 if ((lu[i] > vu_th and se[i] > se_th) or (lu[i] < vu_th and se[i] < se_th)) else 1)
    disagree_before = float(np.mean(disagree) * 100)

    corr = float(np.corrcoef(lu, se)[0, 1])

    # avg LU for correct/incorrect
    vu_correct = float(np.mean(lu[acc == 1]))
    vu_incorrect = float(np.mean(lu[acc == 0]))

    return {
        'vu_threshold': vu_th,
        'se_threshold': se_th,
        'hallu_ratio_before_%': hallu_before,
        'correct_ratio_before_%': correct_before,
        'correct_ratio_ex_refusal_%': correct_ex_refusal,
        'refusal_ratio_before_%': refusal_before,
        'disagree_before_%': disagree_before,
        'corr_before': corr,
        'vu_correct': vu_correct,
        'vu_incorrect': vu_incorrect,
    }

print('Before metrics helper ready.')

In [ ]:
before = compute_before_metrics()
print('Before metrics:')
for k,v in before.items():
    print(k, v)

In [ ]:
def run_eval_scripts_for_current_input(entailment_url: str):
    # Удаляем старые результаты, чтобы не путать сценарии.
    for p in [acc_json, vu_json, vu_most_likely_json, refusal_json, se_pkl]:
        if p.exists():
            p.unlink()

    eval_cmds = [
        [sys.executable, str(CODE_DIR / 'calibration' / 'eval' / 'eval_acc.py'),
         '--dataset', DATASET,
         '--split', SPLIT,
         '--entailment_model', entailment_url,
         '--prompt_type', PROMPT_TYPE,
         '--model_name', MODEL_NAME,
         '--iti_method', str(ITI_METHOD),
         '--max_alpha', str(MAX_ALPHA),
         '--str_process_layers', STR_PROCESS_LAYERS,
        ],
        [sys.executable, str(CODE_DIR / 'calibration' / 'eval' / 'eval_vu.py'),
         '--dataset', DATASET,
         '--split', SPLIT,
         '--entailment_model', entailment_url,
         '--prompt_type', PROMPT_TYPE,
         '--model_name', MODEL_NAME,
         '--iti_method', str(ITI_METHOD),
         '--max_alpha', str(MAX_ALPHA),
         '--str_process_layers', STR_PROCESS_LAYERS,
        ],
        [sys.executable, str(CODE_DIR / 'calibration' / 'eval' / 'eval_vu_most_likely.py'),
         '--dataset', DATASET,
         '--split', SPLIT,
         '--entailment_model', entailment_url,
         '--prompt_type', PROMPT_TYPE,
         '--model_name', MODEL_NAME,
         '--iti_method', str(ITI_METHOD),
         '--max_alpha', str(MAX_ALPHA),
         '--str_process_layers', STR_PROCESS_LAYERS,
        ],
        [sys.executable, str(CODE_DIR / 'calibration' / 'eval' / 'compute_semantic_entropy.py'),
         '--dataset', DATASET,
         '--split', SPLIT,
         '--entailment_model', entailment_url,
         '--prompt_type', PROMPT_TYPE,
         '--model_name', MODEL_NAME,
         '--iti_method', str(ITI_METHOD),
         '--max_alpha', str(MAX_ALPHA),
         '--str_process_layers', STR_PROCESS_LAYERS,
        ],
        [sys.executable, str(CODE_DIR / 'calibration' / 'eval' / 'eval_refusal.py'),
         '--dataset', DATASET,
         '--split', SPLIT,
         '--prompt_type', PROMPT_TYPE,
         '--model_name', MODEL_NAME,
         '--port', entailment_url,
         '--iti_method', str(ITI_METHOD),
         '--max_alpha', str(MAX_ALPHA),
         '--str_process_layers', STR_PROCESS_LAYERS,
        ],
    ]

    for cmd in eval_cmds:
        print('RUN', ' '.join(map(str,cmd)))
        subprocess.run(cmd, cwd=str(CODE_DIR), check=True)

    assert acc_json.exists(), f'missing {acc_json}'
    assert vu_most_likely_json.exists(), f'missing {vu_most_likely_json}'
    assert refusal_json.exists(), f'missing {refusal_json}'
    assert se_pkl.exists(), f'missing {se_pkl}'

print('Eval scripts runner ready.')

In [ ]:
def compute_after_metrics_for_scenario():
    # Load base dataset arrays
    ids, qs, lu, se = load_dataset_base()

    # thresholds from before arrays
    vu_th = before['vu_threshold']
    se_th = before['se_threshold']

    # original accuracy/refusal
    acc_map, refusal_list = load_before_accuracy_and_refusal()
    acc0 = np.array([float(acc_map[i]) for i in ids], dtype=float)
    refusal0 = np.array([int(refusal_list[k]) for k in range(len(ids))], dtype=int)

    # detection
    det_path = CODE_DIR / 'detection' / 'LR_outputs' / DATASET / MODEL_NAME / f'{SPLIT}_verbal_uncertainty_sentence_semantic_entropy.json'
    det_obj = json.load(open(det_path, 'r'))
    detection_res = np.array(det_obj['y_pred'], dtype=int)

    # regenerated outputs
    acc_obj = json.load(open(acc_json, 'r'))  # {id: [0/1,...]}
    re_acc = np.array([acc_obj[str(i)][0] if str(i) in acc_obj else 0.0 for i in ids], dtype=float)

    vu_most = json.load(open(vu_most_likely_json, 'r'))  # {question: score}
    re_vu_most = np.array([vu_most.get(q, -1.0) for q in qs], dtype=float)

    refusal_obj = json.load(open(refusal_json, 'r'))  # {question: 0/1 or -1}
    re_refusal = np.array([int(refusal_obj.get(q, -1)) for q in qs], dtype=int)

    import pickle
    with open(se_pkl, 'rb') as f:
        se_obj = pickle.load(f)
    re_se_list = se_obj['uncertainty_measures']['cluster_assignment_entropy']
    # assume order matches dataset order
    if len(re_se_list) != len(qs):
        raise RuntimeError(f'Length mismatch: re_se_list={len(re_se_list)} vs dataset={len(qs)}')
    re_se = np.array(re_se_list, dtype=float)

    # regeneration flag from input jsonl
    re_generate = []
    with open(input_jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            ex = json.loads(line)
            re_generate.append(bool(ex.get('most_likely_answer','')))
    re_generate = np.array(re_generate, dtype=bool)
    if len(re_generate) != len(qs):
        raise RuntimeError('input_jsonl order length mismatch')

    # hallu + refusal (logic like eval.ipynb)
    re_label = []
    re_acc_final = []
    re_ref_final = []
    for i in range(len(ids)):
        if detection_res[i] == 1:
            ok = (re_vu_most[i] >= vu_th) or (re_acc[i] >= 1e-8)
            re_label.append('ok' if ok else 'hallucinated')
            re_acc_final.append(re_acc[i])
            # if missing refusal (-1), fall back to original
            rr = re_refusal[i]
            if rr == -1:
                rr = int(refusal0[i])
            re_ref_final.append(rr)
        else:
            ok = (acc0[i] == 1)
            re_label.append('ok' if ok else 'hallucinated')
            re_acc_final.append(acc0[i])
            re_ref_final.append(int(refusal0[i]))

    re_label = np.array(re_label)
    re_acc_final = np.array(re_acc_final, dtype=float)
    re_ref_final = np.array(re_ref_final, dtype=int)

    hallu_after = float(np.mean(re_label == 'hallucinated') * 100)
    correct_ratio_ex_refusal = float(np.mean((re_acc_final == 1) & (re_ref_final == 0)) * 100)
    refusal_after = float(np.mean(re_ref_final == 1) * 100)

    # disagreement: use regenerated LU/SE only where detection_res and re_generate are true
    filt_lu = []
    filt_se = []
    filt_acc = []
    for i in range(len(ids)):
        if detection_res[i] == 1 and re_generate[i]:
            filt_lu.append(re_vu_most[i])
            filt_se.append(re_se[i])
            filt_acc.append(re_acc_final[i])
        else:
            filt_lu.append(lu[i])
            filt_se.append(se[i])
            filt_acc.append(acc0[i])
    filt_lu = np.array(filt_lu, dtype=float)
    filt_se = np.array(filt_se, dtype=float)
    filt_acc = np.array(filt_acc, dtype=float)

    disagree = []
    for i in range(len(filt_lu)):
        disagree.append(0 if ((filt_lu[i] > vu_th and filt_se[i] > se_th) or (filt_lu[i] < vu_th and filt_se[i] < se_th)) else 1)
    disagree_after = float(np.mean(disagree) * 100)

    corr = float(np.corrcoef(filt_lu, filt_se)[0, 1])

    vu_correct = float(np.mean(filt_lu[filt_acc == 1]))
    vu_incorrect = float(np.mean(filt_lu[filt_acc == 0]))

    return {
        'hallu_ratio_after_%': hallu_after,
        'correct_ratio_ex_refusal_%': correct_ratio_ex_refusal,
        'refusal_ratio_after_%': refusal_after,
        'disagree_after_%': disagree_after,
        'corr_after': corr,
        'vu_correct_after': vu_correct,
        'vu_incorrect_after': vu_incorrect,
    }

print('After metrics function ready.')

In [ ]:
results = {}
for scen_name, scen_path in SCENARIOS.items():
    if not scen_path:
        continue
    scen_path = Path(scen_path)
    print('\n=== Scenario:', scen_name, '===')
    print('Source jsonl:', scen_path)

    # Copy/overwrite the expected input file for eval scripts
    shutil.copy2(scen_path, input_jsonl_path)

    # Run judges/evaluators
    run_eval_scripts_for_current_input(ENTAILMENT_MODEL_URL)

    # Compute metrics
    metrics = compute_after_metrics_for_scenario()
    results[scen_name] = metrics
    print('Metrics for', scen_name, metrics)

print('\nAll results:')
print(json.dumps(results, indent=2, ensure_ascii=False))

In [ ]:
# Останавливаем vLLM, если мы его стартовали.
# Если vLLM уже был запущен ранее (START_VLLM=False), эта ячейка ничего не сделает.
try:
    if 'vllm_proc' in globals() and vllm_proc is not None:
        vllm_proc.terminate()
        vllm_proc.wait()
        print('vLLM остановлен.')
except Exception as e:
    print('Не удалось остановить vLLM:', repr(e))


## Metrics computed

1) `hallu_ratio_before_%` and `hallu_ratio_after_%`: fraction of questions classified as hallucination (before: `acc==0`; after: `re_label=='hallucinated'`).
2) `correct_ratio_*`: fraction of correct answers (after intervention, refusals are excluded: `refusal==1`).
3) `refusal_ratio_*_%`: refusal fraction.
4) `disagree_*_%`: fraction of examples where `verbal_uncertainty` and `sentence_semantic_entropy` fall on different sides of the thresholds (`vu_th`, `se_th`).
5) `corr_*`: Pearson correlation between `LU` and `SE`.
6) `vu_correct*` / `vu_incorrect*`: mean `verbal_uncertainty` for correct/incorrect answers.

> Thresholds `vu_th` and `se_th` are computed via `best_split()` (as in the paper) but on the available `test.csv`, since `train.csv` may not be present in the checkpoint.